In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 

In [2]:
df = pd.read_csv('10-diamonds.csv')

In [3]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [7]:
df = df.drop(['Unnamed: 0'],axis=1)

In [8]:
df = df.drop(df[df['x'] == 0].index)
df = df.drop(df[df['y'] == 0].index)
df = df.drop(df[df['z'] == 0].index)

In [9]:
df = df[(df["depth"]<75)&(df["depth"]>45)]
df = df[(df["table"]<80)&(df["table"]>40)]
df = df[(df["y"]<30)]
df = df[(df["z"]<30)&(df["z"]>2)]

In [10]:
X= df.drop(["price"],axis =1)
y= df["price"]

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=15)

In [15]:
from sklearn.preprocessing import LabelEncoder

In [14]:
# label_encoder = LabelEncoder()
 # for col in ['cut', 'color', 'clarity']:
  #  X_train[col] = label_encoder.fit_transform(X_train[col])
   # X_test[col] = label_encoder.transform(X_test[col])

 ÖNEMLİ NOT:
 Tek bir LabelEncoder nesnesi kullanmak yerine, her kategorik sütun için 
 (cut, color, clarity) AYRI BİR LabelEncoder oluşturmalı ve saklamalıyız.
 
 SEBEBİ:
 1. Tek nesne kullanılırsa her döngüde hafıza sıfırlanır ve sadece son sütunun 
    (clarity) eşleşme kuralları kalır (üst üste yazma / overwriting problemi).
 2. Modeli dışarı aktarıp (export) yeni verilerle canlıya aldığımızda, 
    'cut' veya 'color' sütunlarına yanlış encoder uygulanır ve model saçmalar.

In [16]:
encoders = {}
for col in ['cut','color','clarity']:
    encoders[col] = LabelEncoder()
    X_train[col] = encoders[col].fit_transform(X_train[col])
    X_test[col] = encoders[col].transform(X_test[col])


In [17]:
X_train.head()

,carat,cut,color,clarity,depth,table,x,y,z
15200,1.15,2,4,2,62.4,54.0,6.71,6.76,4.20
14632,1.11,3,1,2,61.3,58.0,6.66,6.61,4.07
19151,1.21,1,2,5,63.7,58.0,6.67,6.71,4.26
29299,0.30,2,5,5,61.5,58.0,4.28,4.31,2.64
9983,1.00,4,2,2,63.1,57.0,6.37,6.33,4.01


In [18]:
from sklearn.preprocessing import StandardScaler

In [19]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [20]:
from sklearn.svm import SVR

In [21]:
svr = SVR(C=1000,gamma=0.1,kernel = 'rbf')

In [22]:
svr.fit(X_train_scaled,y_train)
y_pred = svr.predict(X_test_scaled)

In [23]:
from sklearn.metrics import r2_score

In [24]:
score = r2_score(y_test,y_pred)
print(score)

0.945219843118452


In [25]:
encoders

{'cut': LabelEncoder(), 'color': LabelEncoder(), 'clarity': LabelEncoder()}

In [26]:
scaler

StandardScaler()

In [27]:
svr

SVR(C=1000, gamma=0.1)

 Artik bunlarinin 3 kullanip disari aktarip bir dosya yapmaya calisacaz 

In [28]:
import pickle 

In [29]:
with open('10-diamond.pkl','wb') as f :
    pickle.dump(
        {
            'model':svr,
            'encoders':encoders,
            'scaler':scaler
        }
    ,f)

 dump kodu ile beraber dosyamizi olusturduk

In [30]:
X_test_scaled

array([[ 0.6196541 , -0.53873069,  0.82581707, ...,  0.83254363,
         0.78315004,  0.76989728],
       [ 1.52658497, -0.53873069,  0.23842797, ...,  1.47548072,
         1.50279799,  1.44915416],
       [-0.56146517, -0.53873069,  0.23842797, ..., -0.45333055,
        -0.42226029, -0.47299831],
       ...,
       [-0.2029111 ,  0.43597985, -0.93635025, ..., -0.02470582,
        -0.08042751, -0.01052553],
       [ 1.5687678 ,  0.43597985,  1.41320618, ...,  1.51119945,
         1.55677159,  1.43470189],
       [-0.94111064, -0.53873069,  0.23842797, ..., -1.02483018,
        -1.06994345, -1.0944461 ]])

In [32]:
pd.DataFrame(X_test_scaled).to_csv('test_data',index = False)


 CANLIYA ALMA / DEPLOYMENT SİMÜLASYONU (PyCharm / .py Mantığı)
> **Not:** Hoca bu kısımdan itibaren PyCharm'a geçti.
> Buradaki amaç, modelimizi (`.pkl`) sanki başka bir yazılımda/canlı sistemde çağırıyormuşuz gibi 
> sıfır hafızalı temiz bir ortamda test etmektir.

In [33]:
import pickle 
import pandas as pd
import numpy as np 

In [35]:
 #yeni bir dosya olusturup ismini requirements yapmaliyiz ve icine asagida kodlari koymaliyiz

# scikit-learn==1.7.0
# pandas==2.3.0
# fastapi==0.115.12
# uvicorn==0.34.3
# Jinja2==3.1.6
# python-multipart==0.0.20

1 - ilk basta veriyi yukleyecegiz 

def main()

  with open('10-diamond.pck','rb') as f  
  saved_data = pickle.load(f)

  model = savaed_data['model']
  x_stes_scaled = pd.read_csv('10-diamond.csv')
  print(model.predict(X_test_scaled))

if __name__ =='__main__':
    main()

bu kodu calistirdik belli basli sayilari verdi bunlar pirlanta fiyatlarini tahmin edebildigimiz gostermektedir     

hocamızla pycharm koldarını olusturduk bır web sayfası yaptık onun kodları elmas dosyasında bulunuyor ve yaptıgımız web sayfasınıda CURSOR ıle gelıstırdık daha guzel bır hale getırdık ılerıde web sayfası ıcın kesıklıkle curso kullanmamız bızım ıcın daha ıyı olacaktır 